# 1-D 'same' convolution: one thread per output

Deep-learning "convolution" is really cross-correlation (the kernel is not flipped). With an odd kernel length $K$ and zero padding of $h = (K-1)/2$ on both sides, the output has the same length as the input:

$$y_i = \sum_{k=0}^{K-1} w_k \, x_{i + k - h}, \qquad x_j = 0 \text{ for } j < 0 \text{ or } j \ge n$$

This is the layer behind 1-D audio and sequence models (and the short causal convolution in Mamba-style blocks, which pads only on the left).

Write `conv1d_same(x, w)`.

## Requirements
- One thread per output element: blocks of **128 threads**, grid $\lceil n / 128 \rceil$, `i = cuda.grid(1)`, threads with `i >= n` do nothing.
- Padding is **implicit**: do not build a padded copy of `x`. Skip (or treat as zero) taps whose input index falls outside `[0, n)`.
- Accumulate in a local `float32` variable and write `y[i]` once.

## Inputs and output
- `x`: `float32`, shape `(n,)`, `n >= 1`. `w`: `float32`, shape `(K,)` with $K$ odd and $K \ge 1$; $K$ may be larger than $n$.
- Return `y`: `float32`, shape `(n,)`. Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Example
`x = [1, 2, 3, 4]`, `w = [1, 0, -1]` gives `y = [0 - 2, 1 - 3, 2 - 4, 3 - 0] = [-2, -2, -2, 3]`.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

TPB = 128

def conv1d_same(x, w):
    """x: float32 (n,), w: float32 (K,), K odd. Returns float32 (n,)."""
    n = x.shape[0]
    d_x, d_w = cuda.to_device(x), cuda.to_device(w)
    d_y = cuda.device_array(n, dtype=np.float32)
    conv1d_kernel[(n + TPB - 1) // TPB, TPB](d_x, d_w, d_y)
    return d_y.copy_to_host()

@cuda.jit
def conv1d_kernel(x, w, y):
    i = cuda.grid(1)
    if i < x.shape[0]:
        acc = float32(0.0)
        for k in range(w.shape[0]):
            acc += w[k] * x[i + k]   # TODO: centre the window and handle the edges
        y[i] = acc


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0, 4.0], "dtype": "float32"}, {"__nd__": [1.0, 0.0, -1.0], "dtype": "float32"}], "e": {"__nd__": [-2.0, -2.0, -2.0, 3.0], "dtype": "float32"}}, {"a": [{"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0], "dtype": "float32"}], "e": {"__nd__": [2.0, 3.0, 3.0, 3.0, 2.0], "dtype": "float32"}}, {"a": [{"__nd__": [2.0, -3.0, 5.0], "dtype": "float32"}, {"__nd__": [4.0], "dtype": "float32"}], "e": {"__nd__": [8.0, -12.0, 20.0], "dtype": "float32"}}, {"a": [{"__nd__": [1.0, 2.0], "dtype": "float32"}, {"__nd__": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0], "dtype": "float32"}], "e": {"__nd__": [14.0, 11.0], "dtype": "float32"}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0, 93.0, 94.0, 95.0, 96.0, 97.0, 98.0, 99.0, 100.0, 101.0, 102.0, 103.0, 104.0, 105.0, 106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0, 114.0, 115.0, 116.0, 117.0, 118.0, 119.0, 120.0, 121.0, 122.0, 123.0, 124.0, 125.0, 126.0, 127.0, 128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 137.0, 138.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 157.0, 158.0, 159.0, 160.0, 161.0, 162.0, 163.0, 164.0, 165.0, 166.0, 167.0, 168.0, 169.0, 170.0, 171.0, 172.0, 173.0, 174.0, 175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 190.0, 191.0, 192.0, 193.0, 194.0, 195.0, 196.0, 197.0, 198.0, 199.0], "dtype": "float32"}, {"__nd__": [0.25, 0.5, 0.25], "dtype": "float32"}], "e": {"__nd__": [0.25, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0, 93.0, 94.0, 95.0, 96.0, 97.0, 98.0, 99.0, 100.0, 101.0, 102.0, 103.0, 104.0, 105.0, 106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0, 114.0, 115.0, 116.0, 117.0, 118.0, 119.0, 120.0, 121.0, 122.0, 123.0, 124.0, 125.0, 126.0, 127.0, 128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 137.0, 138.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 157.0, 158.0, 159.0, 160.0, 161.0, 162.0, 163.0, 164.0, 165.0, 166.0, 167.0, 168.0, 169.0, 170.0, 171.0, 172.0, 173.0, 174.0, 175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 190.0, 191.0, 192.0, 193.0, 194.0, 195.0, 196.0, 197.0, 198.0, 149.0], "dtype": "float32"}}, {"a": [{"__nd__": [-0.5751108527183533, -0.6220993399620056, 0.08025479316711426, 1.2524724006652832, -0.3238966166973114, -1.1016438007354736, -0.7987350821495056, 1.7774511575698853, -0.3504098653793335, -1.1836251020431519, -0.30228927731513977, 0.2982731759548187, 0.287102073431015, 1.8630305528640747, -0.1913972944021225, -1.5456457138061523, 1.5088599920272827, 0.28700950741767883, 0.30000191926956177, -0.6790304780006409, -1.03563392162323, 1.5807695388793945, -1.5177162885665894, -0.6201256513595581, -1.1468671560287476, 1.4436947107315063, 0.3222807049751282, -0.7082746624946594, 0.7131900191307068, 1.9445044994354248, -0.9627354741096497, -0.8935283422470093, 0.39996179938316345, 0.24263527989387512, -1.135886788368225, -0.6383675932884216, -0.019259430468082428, 0.8641139268875122, -1.487180233001709, 0.6720632910728455, 0.5077729821205139, 2.058899164199829, -2.3972198963165283, -0.3414371609687805, -0.39436599612236023, 1.5376944541931152, -1.180694818496704, -0.5536459684371948, 0.26509323716163635, 0.6311063766479492, 0.5079075694084167, -1.2981736660003662, 1.995355486869812, -0.6958400011062622, 0.46154385805130005, 0.9938674569129944, -0.3555768132209778, 0.5626827478408813, -1.5391358137130737, -0.12175184488296509, -0.5469081401824951, -1.5814646482467651, -0.7101176381111145, -1.06833016872406, 0.9790498614311218, -0.8604312539100647, -1.6213092803955078, -0.22810207307338715, -0.3729373514652252, 2.6862642765045166, 1.8651448488235474, 0.17307336628437042, -0.8335529565811157, 2.3971831798553467, 0.2914595603942871, -1.8120982646942139, 0.41106128692626953, -1.8237792253494263, 1.3030167818069458, 0.4196593463420868, -1.2423083782196045, -0.675473690032959, -2.3637309074401855, 1.8078080415725708, 0.06151594594120979, 0.6805455088615417, 1.3650460243225098, -0.4723966717720032, -0.6498909592628479, -0.5547359585762024, 0.2878461182117462, -0.944268524646759, -0.42233189940452576, 0.2600603401660919, 0.11260347813367844, -0.6562219858169556, -1.2211657762527466, -0.6543990969657898, 0.6567620635032654, 1.4043867588043213, 1.2067264318466187, -0.33559632301330566, 0.6703171730041504, 0.39270099997520447, 1.5988413095474243, -0.8714250326156616, 1.2716515064239502, 0.4110044836997986, -0.16683684289455414, 2.6827704906463623, 1.9927374124526978, -1.686310887336731, 0.13997390866279602, 0.8070309162139893, 1.217353343963623, -1.3102936744689941, 0.4841669201850891, 0.24961218237876892, -0.8406502604484558, 0.8671780824661255, -0.5717668533325195, -0.046358268707990646, -1.0642025470733643, 0.5019604563713074, 0.5411787629127502, -0.34771594405174255, -0.9340402483940125, 0.0019263888243585825, 0.2760964632034302], "dtype": "float32"}, {"__nd__": [0.18788649141788483, -0.2759076654911041, 1.1022162437438965, -0.3776001036167145, 0.06896936893463135], "dtype": "float32"}], "e": {"__nd__": [-0.3934566378593445, -0.4709325134754181, -0.34322813153266907, 1.2877920866012573, -0.32659950852394104, -0.4653693735599518, -1.332615852355957, 2.023209810256958, -0.6006231904029846, -0.7392548322677612, -0.16528092324733734, 0.20986059308052063, -0.5393244028091431, 1.995959758758545, 0.01665806770324707, -1.8507404327392578, 1.9659006595611572, -0.5504785180091858, 0.7199482917785645, -0.2772059142589569, -1.5993510484695435, 2.430828809738159, -2.1485188007354736, 0.5648701190948486, -1.9010684490203857, 1.6206375360488892, 0.058048613369464874, -0.7335301637649536, 0.24141575396060944, 2.1153175830841064, -1.0986660718917847, -0.48818036913871765, 0.3365302085876465, 0.37408527731895447, -1.0040711164474487, -0.2777615785598755, -0.4873746931552887, 1.4457248449325562, -2.0999786853790283, 1.263704776763916, -0.8479481339454651, 3.13716721534729, -3.0131895542144775, 0.9268795251846313, -1.4529420137405396, 2.147174835205078, -1.5723986625671387, -0.05213502049446106, 0.019832860678434372, 0.23713171482086182, 1.0633132457733154, -2.2538657188415527, 2.9474997520446777, -1.6671419143676758, 0.675800621509552, 1.0104480981826782, -0.8980426788330078, 1.4778194427490234, -1.9102635383605957, 0.49362295866012573, -0.3102167248725891, -1.4206368923187256, 0.02180437743663788, -1.7077726125717163, 1.4535419940948486, -0.8227589130401611, -1.3052735328674316, 0.3603415787220001, -1.538440465927124, 2.328540802001953, 1.1217199563980103, 0.6609511375427246, -1.5011471509933472, 2.6696815490722656, 0.2158360332250595, -1.9083434343338013, 1.7863390445709229, -2.9271583557128906, 1.772489070892334, 0.18288832902908325, -1.1482266187667847, 0.6943210363388062, -3.330773115158081, 2.541563034057617, -1.0379250049591064, 0.5247758030891418, 1.4619206190109253, -0.5623055696487427, -0.10018789768218994, -0.6947023868560791, 0.6756467819213867, -1.0470255613327026, -0.24132037162780762, 0.13797375559806824, 0.13657674193382263, -0.2895260751247406, -0.8513777852058411, -0.658789336681366, 0.22793757915496826, 0.7649735808372498, 1.2389417886734009, -0.6650069952011108, 1.0201427936553955, -0.47898250818252563, 2.1966183185577393, -1.779677152633667, 1.775766372680664, 0.18645499646663666, -0.933940052986145, 2.2114853858947754, 2.071289300918579, -1.9016282558441162, 0.7731812596321106, -0.015975184738636017, 1.6735795736312866, -1.7940797805786133, 0.9716682434082031, 0.2725931406021118, -1.2713608741760254, 1.4473602771759033, -1.0833110809326172, 0.7060519456863403, -1.4198335409164429, 0.6098496317863464, 0.324929416179657, -0.08543539047241211, -0.8135818839073181, 0.09024699032306671, 0.1282929629087448], "dtype": "float32"}}, {"a": [{"__nd__": [1.7039995193481445, -0.9271090626716614, 0.036712758243083954, 0.3134951591491699, -0.8302896022796631, -1.512558937072754, -1.0677976608276367, -2.342075824737549, -0.00681576132774353, -0.09086039662361145, 0.7608761191368103, 2.056154489517212, 0.18388885259628296, 0.21396051347255707, 0.5320733785629272, -0.5460323095321655, -1.6737757921218872, -0.8422238826751709, 0.16125139594078064, -0.03324762359261513, 0.028213100507855415, -0.6845755577087402, 0.04171673208475113, -0.3728910982608795, -0.18530532717704773, -1.4223036766052246, -0.9564550518989563, -0.008256782777607441, 1.6381862163543701, 0.08179155737161636, 0.6347742080688477, 2.0460774898529053, 0.2866290807723999, -0.34788164496421814, -2.157292127609253, -0.3869590759277344, 1.3113657236099243, -1.1076993942260742, -0.40452972054481506, -0.21416698396205902, 0.22509071230888367, -1.16325044631958, 0.854256272315979, 0.6166239976882935, -1.1302826404571533, -3.127065658569336, 0.6407890319824219, 0.13488219678401947, 0.013956756331026554, -2.0137691497802734, -0.9281610250473022, -0.33284303545951843, 0.22437714040279388, -0.14916932582855225, 1.0076525211334229, -1.2744909524917603, -1.3715311288833618, -0.10365680605173111, 1.2813814878463745, -1.4707409143447876, -1.6125315427780151, 1.8221155405044556, -0.8336336612701416, 0.08261288702487946, 1.4714751243591309, -1.1406373977661133, 0.07833101600408554, 0.5096641182899475, -0.20809896290302277, -0.3918152153491974, -0.7730690836906433, -0.48302340507507324, 1.1056818962097168, -2.4598121643066406, 0.4029462933540344, -0.059381432831287384, 0.021965838968753815, 0.015681955963373184, -0.0732952207326889, 0.6357713937759399, -2.201606512069702, 0.7973386645317078, 0.7211654186248779, 0.042687561362981796, -0.11891114711761475, 0.9787241220474243, 2.1934239864349365, 1.7382320165634155, 1.446631669998169, -1.9837299585342407, 1.8644628524780273, 0.10962424427270889, -0.6747642755508423, 0.33558332920074463, 1.3060853481292725, -1.8441483974456787, 0.8336985111236572, -0.5506142377853394, 0.120498888194561, -0.3168739974498749, 0.806750476360321, 0.0955682173371315, -0.7549429535865784, -0.9325592517852783, -0.5712764859199524, 0.6240311861038208, -0.7991223335266113, 0.8686080574989319, 1.3323843479156494, 0.2901344895362854, -2.1413121223449707, 0.4487391412258148, 0.5897481441497803, 1.0043580532073975, 0.3219127058982849, 1.536711573600769, 1.6234463453292847, 0.13040587306022644, 0.9018996953964233, 2.087416887283325, 0.7584556341171265, 0.5292584896087646, 2.196117877960205, -2.0937347412109375, 1.5808693170547485, -0.2715640664100647, -2.253720283508301, 1.0409348011016846, 1.5835213661193848, -0.009889334440231323, 0.636734127998352, 1.220943570137024, -0.4053606390953064, 0.39383742213249207, 0.8466071486473083, -0.6759977340698242, -0.046854645013809204, -0.9118230938911438, -1.0202223062515259, -0.09064143151044846, 0.45227310061454773, 0.2547551691532135, 0.6969286203384399, 0.8075539469718933, 0.7773864269256592, -0.569456160068512, -0.3313841223716736, -1.3684386014938354, 1.6850379705429077, 0.9526498913764954, -0.30506929755210876, -0.05584339797496796, 0.9053038954734802, -0.9154143929481506, -0.46237698197364807, -0.8937602043151855, -0.029957322403788567, 1.362604022026062, -0.531647264957428, 1.301157832145691, 0.3749905228614807, -0.2761062681674957, 0.8840526342391968, -1.1920039653778076, 0.16361215710639954, 0.13526682555675507, 2.3323962688446045, -0.42120325565338135, 0.2857210040092468, -1.265523910522461, -1.018675446510315, 0.7163978219032288, 1.804069995880127, 1.3331218957901, -1.5458929538726807, -1.0635044574737549, -1.3488882780075073, -0.09011421352624893, 1.1555556058883667, 0.9346250295639038, 0.09768843650817871, 0.026761287823319435, -2.4630837440490723, -3.7556352615356445, -0.40561816096305847, -1.9417740106582642, -0.19895000755786896, -1.2582067251205444, 0.6186527609825134, -0.7773506045341492, -1.1422842741012573, -0.2892606258392334, 0.9737468957901001, -0.5923742651939392, 1.270981788635254, -1.456227421760559, -1.479676604270935, -0.5222333073616028, -0.21156245470046997, 0.4553145170211792, -0.27932193875312805, -0.24260035157203674, -1.7016814947128296, 1.133506417274475, -0.6287630796432495, 0.10725092887878418, -0.0028553626034408808, 1.9249249696731567, -1.925808072090149, -0.1637767106294632, 1.373853087425232, -2.5317134857177734, -0.6627234220504761, -0.21431301534175873, 0.0342167429625988, -1.397168517112732, 0.712325394153595, -1.3897154331207275, 1.2405282258987427, -0.24492144584655762, -1.4019410610198975, 1.3274552822113037, 0.14487133920192719, 0.46511080861091614, 0.8170512914657593, -0.4699631929397583, -0.6128697991371155, 0.34505030512809753, 1.4052655696868896, 0.38600829243659973, -0.6611413359642029, 1.7599655389785767, -0.8619813919067383, -0.46602001786231995, -0.4040693938732147, -0.18081897497177124, 0.4080273509025574, 0.5335169434547424, -0.8930636644363403, -0.8360075354576111, 0.9019038677215576, 0.8679361939430237, -0.945877730846405, 1.1516497135162354, -0.13947200775146484, 0.4125814735889435, 0.3334103524684906, 0.3607948124408722, -0.4112023711204529, 0.12358555197715759, -0.5242670774459839, 0.141609787940979, -0.3291168510913849, -0.5038905739784241, -0.22051915526390076, 0.45668211579322815, 0.1872207522392273, -0.1277657002210617, -0.3716748356819153, 0.4589802324771881, 1.7319600582122803, 0.20896857976913452, 0.5067608952522278, -0.814583420753479, -0.4397413730621338, -0.23853594064712524, 1.783062219619751, 0.8487232327461243, 0.8009360432624817, -0.03016294538974762, 0.20262695848941803, 0.7316219806671143, 0.13571201264858246, -1.1491328477859497, -0.03386583924293518, -0.022420654073357582, -1.5808076858520508, -2.0406875610351562, -0.16074059903621674, -0.2382230907678604, -0.0919969230890274, 0.20949143171310425, 0.4867090880870819, 1.1321930885314941, -1.1604186296463013, 0.4421330988407135, 1.4281100034713745, 1.2919126749038696, -1.0844076871871948, 0.343471884727478, 0.32247331738471985, -0.4928308427333832, -1.2219030857086182, -0.30740734934806824, 1.2195197343826294, -0.34377554059028625, -0.14662465453147888, -0.30955564975738525, 1.1388440132141113, 0.08100093901157379, -1.8929991722106934, -2.448110818862915, -0.08619803935289383, -0.5511788725852966, -0.5856081247329712, 1.508076786994934, 1.2338659763336182, 0.25336164236068726, 0.9359338879585266, 1.4780319929122925, 0.10957138240337372, -0.4523899555206299, -1.6200357675552368, 0.7495870590209961, 1.5404177904129028, -1.0424039363861084, -0.5023291707038879, -0.7794713973999023, 0.6361508369445801, -0.31177660822868347, 1.3369178771972656, -1.2839769124984741, 2.7358109951019287, 1.472993016242981, -0.34527212381362915, -0.8636570572853088, -1.1184321641921997, 0.32463520765304565, 0.050203222781419754, -0.33830204606056213, -0.40087926387786865, -0.08004536479711533, -0.2184370756149292, -0.5854184627532959, 1.203322410583496, 1.3499035835266113, -0.1711445450782776, 0.5551202893257141, -0.5620043873786926, 0.6773493885993958, -1.8711200952529907, 0.25098443031311035, 0.43995869159698486, 0.03218080848455429, -0.5131973624229431, 0.22424888610839844, 0.027973711490631104, 1.640076756477356, -1.2605881690979004, -1.8172537088394165, 0.29553937911987305, 0.18023717403411865, -0.2182295173406601, -0.35309115052223206, 1.1657260656356812, 1.8300788402557373, -0.43703171610832214, 1.5051778554916382, 0.6419427990913391, 0.906775712966919, 0.02904309704899788, 0.04132942482829094, 1.4869437217712402, -0.7099602222442627, 0.2050381302833557, 0.7698156833648682, -0.5464431047439575, -0.7885810732841492, -0.5732165575027466, 1.156829833984375, 0.18889231979846954, -0.3337419927120209, -0.7620149254798889, 0.6620162725448608, -0.002573989098891616, -0.018080055713653564, -0.93807053565979, -0.32610535621643066, -0.79270339012146, 0.8018386363983154, 0.012133590877056122, -1.526695966720581, 0.5308404564857483, -0.326871782541275, 2.3235340118408203, -0.1345292031764984, 0.4267904758453369, -0.30663713812828064, -0.36949026584625244, -0.0023140935227274895, 0.13158540427684784, 0.5424279570579529, 2.0096421241760254, 0.7227950096130371, 1.0979255437850952, -1.8062009811401367, 1.888074278831482, -2.2420337200164795, 0.57557213306427, 1.0283814668655396, 0.14197611808776855, -1.009402871131897, 2.65446400642395, 1.0380655527114868, 0.19148442149162292, 2.004091501235962, 2.717036724090576, -0.773246705532074, 0.41269078850746155, -2.0533502101898193, -1.0045380592346191, 0.13746201992034912, 1.639771819114685, 0.804617166519165, 3.1695408821105957, 0.7190817594528198, -2.150372266769409, -0.6716158390045166, 1.6067826747894287, 0.9580290913581848, -0.18277299404144287, 1.2199547290802002, 0.8364923000335693, 0.8161497712135315, -2.064229965209961, 0.2148110270500183, -1.679311990737915, -1.324296236038208, 1.1479307413101196, 0.044189196079969406, 0.40794843435287476, 1.0839489698410034, -0.7848241925239563, -0.9665155410766602, -0.11950181424617767, 0.015920277684926987, -0.5207091569900513, -0.15951204299926758, 1.8374704122543335, -0.3847815692424774, -0.09513013064861298, -1.0045592784881592, -0.7928573489189148, 0.535435140132904, 0.0006574479630216956, 0.464810848236084, -0.46216917037963867, 1.5194040536880493, 1.776263952255249, -0.0837031826376915, 1.4359418153762817, 1.9136861562728882, -0.1197023093700409, -0.8053112030029297, 1.0413968563079834, 1.1511213779449463, -1.7409895658493042, -2.125978946685791, -0.7236580848693848, 1.7815736532211304, 0.44455164670944214, 0.5171822905540466, -1.5139784812927246, -0.5204287171363831, -1.6200685501098633, -0.9323086142539978, 0.769701361656189, 0.06755000352859497, -0.005576957482844591, -0.373041033744812, -0.36437684297561646, 0.1350802630186081, -0.8782507181167603, 0.554478645324707, -0.4070926904678345, 1.5355894565582275, -2.316423177719116, -0.023898903280496597, -0.7129239439964294, 0.19177865982055664, -0.4523015022277832, -0.9485366940498352, 0.2333308756351471, 0.9598620533943176, 0.5296594500541687, 1.2409391403198242, 0.14946244657039642, 0.8897521495819092, -0.7591018080711365, -0.7140188813209534, 0.5107314586639404, -1.4142574071884155, -1.1264681816101074, 0.29381391406059265, -0.9699723720550537, 1.4402837753295898, 1.0236090421676636, -1.2600820064544678, 0.5666437149047852, -0.6094561815261841, -0.18313735723495483, -0.2746879756450653, -0.5074014067649841, -0.2738972008228302, -0.9018896222114563, -2.235323905944824, 0.17511633038520813, 0.05150187388062477, -0.7321274280548096, 0.3975478708744049, -0.6501725912094116, 1.1749104261398315, -0.5801412463188171, 0.8607495427131653, 1.0728192329406738, 1.7365370988845825, -1.1464447975158691, 0.06588497757911682, 1.7694555521011353, -1.7598772048950195, -0.7848720550537109, -0.937606930732727, -0.33245277404785156, 0.8306780457496643, -0.42991018295288086, 0.7424032092094421, 0.4909707009792328, -1.4742592573165894, 0.22782662510871887, -0.19205676019191742, 0.6810990571975708, 0.7037540674209595, -0.5282094478607178, -0.9056440591812134, 0.6128144860267639, 1.8146014213562012, -0.13609479367733002, 0.2006954699754715, -1.269590973854065, 2.7545485496520996, 0.346172034740448, -1.0281572341918945, 0.19182375073432922, 1.4355932474136353, 0.9111127257347107, 0.3278542160987854, -0.32307562232017517, -1.675796389579773, -0.9470299482345581, 0.19931693375110626, 0.5616506934165955, 0.10801778733730316, 0.281220018863678, 0.6241155862808228, -1.204226016998291, -1.1058229207992554, -0.825494647026062, 1.0024205446243286, 0.265727162361145, 0.33345234394073486, -0.7027530670166016, -0.5073084235191345, -0.02733716182410717, -0.8512247204780579, -1.0104731321334839, 0.031705260276794434, -0.5436646342277527, -0.22164084017276764, 1.3011020421981812, 0.9081119894981384, -0.7040321826934814, -0.01430823840200901, -0.0067441510036587715, -0.536571741104126, 0.4552253782749176, -0.1824512630701065, -0.6687685251235962, 0.5903692841529846, -0.7891640067100525, -0.887921929359436, 0.9690476655960083, 0.11444864422082901, -0.47851938009262085, 0.8299378752708435, -0.6940078139305115, 0.0604582317173481, -0.008240303955972195, 0.08793964982032776, -0.5629070401191711, 0.4425913095474243, 2.360111713409424, 1.2105978727340698, 0.8020839691162109, 0.025164863094687462, -1.1248944997787476, 2.294262170791626, -0.6562293767929077, 1.1041693687438965, 1.2196708917617798, 0.8686445951461792, -0.44120651483535767, -2.9410946369171143, -1.3713141679763794, -0.9469305276870728, -0.8042502403259277, 1.5097827911376953, 1.3149679899215698, 0.721792459487915, 1.8710649013519287, 0.6784298419952393, 0.005638600327074528, -0.28007471561431885, -0.18632180988788605, -0.3707781136035919, 0.9556924104690552, 1.3122694492340088, -0.5908921957015991, 0.6314727067947388, -0.682283341884613, 1.2612968683242798, -1.3776520490646362, 0.2452700436115265, -0.14413493871688843, -1.7426033020019531, -2.138718843460083, -1.6006274223327637, -0.5430935621261597, 0.019353670999407768, 1.2129688262939453, 0.7329095005989075, -0.8451103568077087, 0.7334153056144714, 2.1805531978607178, 0.033947646617889404, -0.16370531916618347, -0.6439959406852722, -0.47970640659332275, 1.1191664934158325, -1.2368439435958862, 0.6823976635932922, 0.7326217889785767, -0.48637133836746216, 2.761571168899536, -0.36759963631629944, -1.4034829139709473, 0.6541600823402405, -1.489722728729248, 0.3304399251937866, -0.3864128887653351, -0.0033764445688575506, 0.8983046412467957, 0.15843993425369263, 0.4349828064441681, -0.5539475083351135, -0.040841132402420044, -1.01522958278656, -0.610543429851532, 0.039135370403528214, 1.2741191387176514, 0.7708967328071594, 0.5685409903526306, 0.4082823097705841, -1.1063240766525269, 0.7135142683982849, 0.9277806878089905, 2.12015438079834, 0.17893363535404205, -1.7848228216171265, 0.3314814269542694, 1.3478598594665527, -1.1082887649536133, -0.041390784084796906, 0.5610737800598145, -0.2858608067035675, 0.9455414414405823, 0.24865619838237762, 0.9891548156738281, -1.9926987886428833, 0.4368518590927124, 0.05347162112593651, 0.4232405126094818, -0.7156969904899597, 0.8340616226196289, -0.9184725284576416, 1.239680290222168, -0.7291867136955261, 0.9535265564918518, -0.4712437391281128, 0.1327398121356964, -0.20407238602638245, 0.007546817883849144, 0.46988484263420105, -1.7791141271591187, 1.8436990976333618, 0.5076571106910706, 0.15622684359550476, -0.1787347048521042, -1.4044378995895386, -1.3141940832138062, -0.3467075228691101, -0.6778308749198914, 0.5042168498039246, 0.9411513805389404, 0.8297697305679321, 0.04062587022781372, -0.762237548828125, 0.2515220046043396, 0.9827035069465637, -0.6400185227394104, 1.5365962982177734, 0.49757131934165955, -0.2806110084056854, 0.15300533175468445, -0.04981544613838196, 0.8099949955940247, -1.704089879989624, 1.093906283378601, -1.0029382705688477, 1.9835643768310547, -0.15573526918888092, -1.780033826828003, 2.044116735458374, -1.2989665269851685, -1.528605341911316, -0.5348277688026428, 0.7873275279998779, 0.32418543100357056, 0.940528154373169, 0.23573490977287292, 1.3333961963653564, 1.0210940837860107, 1.2189642190933228, 1.5287361145019531, -1.0475651025772095, 0.8067566156387329, 0.40913891792297363, 0.041249603033065796, -0.5763812065124512, 0.17996883392333984, 0.43354469537734985, -2.5432138442993164, 0.23887676000595093, 0.09114256501197815, -0.06761027872562408, -0.07415784150362015, -0.8137871623039246, -0.7301993370056152, -0.7931565046310425, -1.8928335905075073, -0.6982592940330505, 0.4627630412578583, 0.8836933374404907, -0.6298798322677612, -1.1116490364074707, -1.7122666835784912, 0.826440691947937, 0.5355619788169861, 0.4245729446411133, -1.404110074043274, 0.5045372843742371, 0.0509134978055954, 0.14249387383460999, 0.26671433448791504, 0.9509541392326355, -1.915079116821289, -0.6113733649253845, 1.2670185565948486, -1.3225369453430176, 0.6099339723587036, -1.3422547578811646, -1.3125731945037842, -0.7801333665847778, 0.11329793184995651, 2.1217780113220215, 0.35869336128234863, -0.41559481620788574, -1.0486289262771606, -0.3109790086746216, -0.6942586302757263, -0.5736997723579407, 0.5969831347465515, -0.6774123907089233, 0.19439122080802917, -0.6752263307571411, 1.4728772640228271, 1.165244221687317, 0.07927460223436356, -1.6458959579467773, -1.572985053062439, -0.8210834264755249, -0.6251058578491211, 2.1264736652374268, 0.599018931388855, -0.010051819495856762, 0.645592212677002, 1.3591567277908325, -0.47128432989120483, -0.8036170601844788, 0.980082631111145, 0.34491127729415894, -2.1830942630767822, -0.15093070268630981, -1.086846113204956, 1.6187567710876465, -0.1688379943370819, -1.5250176191329956, 1.2095894813537598, -0.0773908719420433, -0.3795539140701294, 0.14153139293193817, -0.7751707434654236, -1.2047871351242065, -1.089112639427185, -0.024857202544808388, 0.8253220915794373, -0.5931099057197571, -0.7528849840164185, 2.3557515144348145, 0.5599976778030396, -0.7828351259231567, -0.8743816018104553, 0.5927574634552002, 0.9502102732658386, -1.1287977695465088, 0.2793445587158203, -1.2769628763198853, -0.1640782207250595, -0.18836919963359833, 0.7845559120178223, 1.1235997676849365, -0.3528037667274475, -1.7929085493087769, 0.6808223128318787, -1.911643385887146, 0.2098884880542755, 1.6308698654174805, -0.4055436849594116, 0.1972818672657013, -0.8767741918563843, -1.493913173675537, 0.8362693190574646, -0.751765251159668, 1.5686607360839844, -0.6246622800827026, -0.712843656539917, -0.11055511236190796, 0.18332280218601227, -0.4091421067714691, 2.2139594554901123, -1.2039939165115356, 0.2902510166168213, -0.7853948473930359, 0.963390588760376, 0.24498921632766724, -0.7333970665931702, -0.5106937885284424, -0.3439338803291321, -0.18375235795974731, -1.6429790258407593, 1.483485460281372, 1.891617774963379, 2.1509294509887695, 2.276402711868286, -0.7855456471443176, -1.516486406326294, -0.10797391086816788, -0.0099988654255867, 1.1942918300628662, -2.0915417671203613, -0.3655332922935486, -1.0562334060668945, -1.4839489459991455, -0.39876049757003784, -1.011936902999878, -0.21525244414806366, 1.7092034816741943, 0.039312031120061874, 0.2754184901714325, 0.512164294719696, -1.0770021677017212, 0.48643824458122253, 0.21477316319942474, -0.2158859223127365, 1.5965147018432617, -1.3600642681121826, 0.3036118745803833, -0.6692184209823608, 0.4124879539012909, -1.245789885520935, 0.2856369912624359, 0.003909999970346689, -0.9627461433410645, -0.5882521867752075, 1.2172127962112427, -0.2895069718360901, 0.3791520595550537, 0.45702555775642395, 1.1119953393936157, -0.22153733670711517, -0.19674086570739746, -0.5268100500106812, 0.46312248706817627, 0.6850173473358154, 0.18088366091251373, -0.1791687160730362, -0.13015474379062653, -0.7210301160812378, 0.1573445051908493, -0.46500834822654724, 0.6713801026344299, 0.09778929501771927, 0.40586158633232117, 0.013217995874583721, -1.4629106521606445, -2.0915727615356445, -1.6174812316894531, -0.11045776307582855, 0.7214148044586182, 0.09328512102365494, -0.4948239326477051, -1.3242462873458862, 0.8038095235824585, 1.8307995796203613, -0.8362058401107788, 1.9452688694000244, -0.9678614139556885, -0.7293224930763245, -1.9333926439285278, -0.6076006293296814, -0.17424604296684265, -0.29945650696754456, -0.34812062978744507, 1.927608609199524, -0.09165346622467041, -0.8544613718986511, 0.1776444911956787, 0.23967385292053223, 2.1515142917633057, -1.8065003156661987, -0.46489110589027405, 1.181403398513794, 0.5849623680114746, 0.4230802059173584, -1.4067637920379639, 0.4020727574825287, -0.5849660634994507, -0.4390288293361664, -0.49669888615608215, 0.8790644407272339, 0.5647662281990051, -0.3776668310165405, 0.918245792388916, 0.17151102423667908, -1.1585702896118164, -1.5699764490127563, 0.5153566002845764, 0.09950517863035202, 1.3908170461654663, -0.5325688719749451, 0.4652842879295349, -0.6557050347328186, 0.3835982084274292, 0.6154147982597351, -0.8319146037101746, -0.4993990957736969, 0.059549037367105484, -0.6022202968597412, 2.6161184310913086, -2.239699363708496, 0.5195285677909851, 0.16268625855445862, -2.032396078109741, 0.31570935249328613, -0.6360311508178711, -1.0391037464141846, 0.4262250065803528, 0.300326943397522, 0.6371253132820129, 0.6296554207801819, -0.8911630511283875, -0.4018917679786682, -0.2163861244916916, 0.8260416388511658, 0.7595458030700684, 0.32579997181892395, 1.4857637882232666, 0.7279057502746582, 0.7662917375564575, -0.6353318095207214, 0.13970765471458435, -0.35644298791885376, -1.8795849084854126, 0.5634946823120117, 0.886807918548584, 0.7486755847930908, 1.5518428087234497, 0.9335556030273438, -0.27263587713241577, 0.2647702395915985, -0.6839605569839478], "dtype": "float32"}, {"__nd__": [-0.7190049290657043, -0.5181072950363159, -1.0638518333435059, -0.26949742436408997, 1.608838438987732, 1.7757959365844727, -0.04901319742202759, -0.9244117736816406, -0.9283598065376282], "dtype": "float32"}], "e": {"__nd__": [1.5743128061294556, 0.27076518535614014, 1.4830416440963745, 2.3590052127838135, -2.6664769649505615, -3.58638596534729, -5.396886825561523, -4.286203861236572, 0.8667555451393127, 4.869810581207275, 6.18761682510376, 5.218245029449463, 1.3967320919036865, 1.079101324081421, -1.2683972120285034, -5.872352600097656, -4.853777885437012, 0.14489486813545227, 2.7010717391967773, 2.450117349624634, 0.8216419219970703, 1.0324782133102417, 1.6725102663040161, 0.7588814496994019, -3.899651050567627, -4.659372329711914, -1.5549969673156738, 2.540503978729248, 2.4820897579193115, 2.3020167350769043, 5.89208984375, 5.070413589477539, -3.3575596809387207, -7.1971869468688965, -4.550320625305176, 1.6650660037994385, 2.5250442028045654, -0.192041277885437, -0.10489368438720703, -0.3637186288833618, -1.1470305919647217, 4.736380577087402, 5.296421051025391, -0.5347352623939514, -8.17465591430664, -2.000645399093628, 5.1044816970825195, 4.803785800933838, -1.6939359903335571, -3.1715519428253174, -2.895646333694458, 2.410245895385742, 4.608572959899902, 5.1983771324157715, -1.0273265838623047, -4.289877891540527, -0.40943384170532227, 3.290529727935791, 0.041219353675842285, -3.2331809997558594, -0.6871611475944519, 2.555610179901123, 0.7802205085754395, 2.4354302883148193, 1.1400041580200195, -2.5275120735168457, 1.41971755027771, 2.0032618045806885, -2.2603600025177, -0.4246053695678711, -0.24928870797157288, 1.3561012744903564, -1.269278883934021, -2.375706911087036, 0.8880270719528198, 1.7029505968093872, 1.58973228931427, 2.7759859561920166, -0.5741313695907593, -3.597729444503784, -2.207815408706665, 1.7063143253326416, 0.151444673538208, -4.19084358215332, -1.119496464729309, 4.928519248962402, 5.969698905944824, 2.0359609127044678, -3.9851138591766357, -3.6529719829559326, -1.7774372100830078, -0.9250321388244629, -1.6483006477355957, 3.215658187866211, -1.5872300863265991, -1.7149063348770142, -0.6796283721923828, -0.6715855002403259, -0.5180249214172363, 3.9290740489959717, 2.540109634399414, -0.7392410635948181, -3.4840877056121826, -2.701420783996582, -1.3867579698562622, -0.4902171790599823, 3.3767640590667725, 5.830117225646973, 2.504940986633301, -6.15311861038208, -5.2754621505737305, -1.0506404638290405, 0.837662935256958, 0.754879355430603, 2.6184275150299072, 0.7937966585159302, -1.535524606704712, -2.4446587562561035, -0.1972658634185791, 2.265045404434204, -0.2365647554397583, 0.6585571765899658, -0.5985243320465088, -2.479590654373169, -2.8523807525634766, -5.6616950035095215, -4.539838790893555, 4.347733974456787, 2.8676679134368896, 0.8928239941596985, 1.4615460634231567, -0.6598331332206726, -1.4657493829727173, 1.5475224256515503, 1.1886606216430664, -1.4138180017471313, -2.6117501258850098, -3.918846845626831, -2.6698951721191406, 1.006194829940796, 1.2935662269592285, 2.5766496658325195, 3.581949472427368, 3.6563565731048584, -1.4598671197891235, -5.492508411407471, -5.236279010772705, 0.7895660996437073, 4.086221694946289, 2.5894532203674316, -0.46097612380981445, 1.9994378089904785, -0.6579524874687195, -4.198798179626465, -3.563732624053955, -1.6047992706298828, 1.402767539024353, 2.950669527053833, 1.3315603733062744, 2.413980007171631, 0.5501716732978821, -1.2822774648666382, -3.60922908782959, -4.473234176635742, -0.2325994074344635, 6.255374908447266, 4.877470970153809, 0.1681717038154602, -6.630309104919434, -7.720008850097656, -1.674025297164917, 8.482917785644531, 8.921843528747559, 0.9716675281524658, -7.216289043426514, -8.489243507385254, -3.3980960845947266, 3.083444595336914, 8.897724151611328, 8.687291145324707, 3.7090511322021484, -3.523719549179077, -10.05859375, -5.402641296386719, 0.04613792896270752, 2.095797538757324, 3.865936040878296, 5.466779708862305, 0.8827173113822937, -0.9440818428993225, -2.68520450592041, 3.1887855529785156, 4.423309326171875, 4.424047946929932, 0.3054269552230835, -5.180820465087891, -4.8120598793029785, 0.9606260061264038, 3.8414387702941895, 3.1954126358032227, 0.17593663930892944, -2.9100422859191895, -0.5117800831794739, -0.5485314130783081, 1.0188236236572266, 2.025067090988159, 3.6602838039398193, 0.16285395622253418, -0.6201017498970032, 1.5073997974395752, -0.9890217781066895, -4.556301593780518, -0.12796998023986816, 2.6940438747406006, -1.2412492036819458, 0.5483090281486511, 1.07294762134552, 1.4734911918640137, 0.5839817523956299, -1.7347321510314941, -2.139655351638794, 3.045994758605957, 2.3910348415374756, 1.9196498394012451, -1.0728065967559814, -5.263245105743408, -1.2729078531265259, 1.9163621664047241, 2.3670578002929688, 0.49981167912483215, 1.5726954936981201, 0.656378984451294, -3.387521743774414, -3.9059548377990723, -0.04915904998779297, 1.7950366735458374, 2.922980785369873, -1.6992443799972534, -3.0834453105926514, -0.3890058398246765, 2.7091338634490967, 0.1315716803073883, -0.27889326214790344, 0.40713340044021606, 0.13790157437324524, 0.18196892738342285, 1.6903207302093506, -1.0871937274932861, -0.80717933177948, -0.7094763517379761, 0.07216188311576843, -0.5885890126228333, -1.258259892463684, -1.1583033800125122, 1.699684739112854, 1.6551315784454346, -1.330398678779602, -2.742856740951538, -0.7742264270782471, 3.9022932052612305, 4.497237205505371, 1.2333612442016602, -3.9139325618743896, -6.102169513702393, -4.419730186462402, 2.6008970737457275, 4.766116619110107, 2.5094757080078125, -1.2631067037582397, -0.6542824506759644, 0.14655670523643494, 0.47877591848373413, -1.305327296257019, 0.5497939586639404, 1.6594043970108032, -1.7382146120071411, -5.3137407302856445, -2.3726894855499268, 0.9450414180755615, 0.9915568828582764, 2.6599321365356445, 3.636859893798828, 1.396355152130127, -2.9154295921325684, -2.204625129699707, 2.573056936264038, 4.206857204437256, -0.7726834416389465, -0.8227523565292358, 0.4243505299091339, -1.7795355319976807, -4.575220584869385, -1.7261073589324951, 2.550384283065796, 1.9938703775405884, -0.9404380321502686, 0.6741673946380615, 5.537424087524414, 3.9526076316833496, -2.174348831176758, -7.30306339263916, -4.866850852966309, -1.8154844045639038, 0.17378868162631989, 3.442021131515503, 4.916650772094727, 1.485228419303894, 1.078708291053772, 4.306028366088867, 1.150997281074524, -5.083100318908691, -6.366688251495361, -1.354180097579956, 4.9789018630981445, 2.4588534832000732, -2.877478837966919, -3.7578630447387695, -0.24567711353302002, -0.7716293334960938, -0.29350149631500244, -1.1321228742599487, 4.043530464172363, 7.498702049255371, 2.691688299179077, -5.984528541564941, -5.092100620269775, -2.670619249343872, 1.4129225015640259, 1.5745220184326172, 0.3347231149673462, -0.6786112785339355, -2.645033836364746, -1.9517468214035034, 1.366668939590454, 5.069935321807861, 2.2005832195281982, 0.6653196811676025, -0.23392951488494873, -1.7822041511535645, -4.002884387969971, -1.8887255191802979, 1.1254220008850098, 2.5329535007476807, -2.322146415710449, -0.036045074462890625, 2.878103256225586, 4.577281951904297, 0.0452621728181839, -5.452607154846191, -3.4589571952819824, 1.0113993883132935, -1.529296636581421, -0.8363556265830994, 1.4395374059677124, 3.180818557739258, 0.7041661143302917, -0.3203992247581482, 1.2726365327835083, -0.5040613412857056, -2.07073974609375, -0.8797365427017212, -0.549734890460968, -0.05627572536468506, -0.6903309226036072, 1.5519659519195557, -0.33735495805740356, -4.623027324676514, -2.4755682945251465, 2.3863461017608643, 3.02579927444458, 0.23852209746837616, -2.321023464202881, 0.5391260385513306, 1.8620792627334595, 1.716458797454834, -1.7593624591827393, -2.589432954788208, -0.7681413888931274, 2.1635308265686035, 2.258723258972168, -3.101848840713501, -3.7316629886627197, 0.44089189171791077, 4.39527702331543, 4.4104790687561035, 1.4444221258163452, -2.607760429382324, -2.715280055999756, -4.9444427490234375, -2.023993968963623, -0.3652750253677368, 5.446744918823242, 4.36046838760376, 2.3519678115844727, -2.70377516746521, -3.4259886741638184, -4.555142879486084, -1.505368709564209, -0.04397106170654297, 1.089097023010254, -3.9169373512268066, 1.1829304695129395, 0.9039561748504639, -0.33640599250793457, 1.3826121091842651, 7.806328773498535, 2.6184206008911133, -3.31941556930542, -8.44066047668457, -9.494166374206543, -6.577284812927246, 2.2934207916259766, 7.041969299316406, 10.907715797424316, 4.300527095794678, -7.662132740020752, -10.608537673950195, -1.6439213752746582, 2.20816969871521, 0.5045846700668335, 2.9197611808776855, 3.797389030456543, 2.4707250595092773, -1.6987425088882446, -4.309248447418213, -5.296298980712891, -4.415408611297607, -0.7709407806396484, 5.148225784301758, 4.1802778244018555, 4.28448486328125, 0.6937446594238281, -4.078824996948242, -2.323310613632202, -1.4563539028167725, -1.5449122190475464, 0.4208465814590454, 4.925971984863281, 4.6168646812438965, -0.5700560808181763, -3.787715435028076, -3.8848183155059814, -1.07526695728302, 1.464818000793457, -0.916394829750061, -1.1484087705612183, 0.7785319089889526, 1.7436119318008423, 0.8217041492462158, 0.9898844957351685, 3.1699936389923096, -1.436307430267334, -4.398675441741943, 1.392929196357727, 4.762453079223633, -2.858309268951416, -9.515230178833008, -6.399789333343506, 3.98502779006958, 8.044140815734863, 6.3359246253967285, 0.4205681085586548, -4.148288726806641, -6.0962114334106445, -3.1936981678009033, 1.61786949634552, 5.3746337890625, 2.3264544010162354, 0.8477253913879395, -0.7555463910102844, -0.9198912382125854, -1.9693270921707153, 0.8711118698120117, 2.807389974594116, 3.846325397491455, -1.2830805778503418, -3.128195285797119, -1.2122852802276611, 1.8438458442687988, -1.2413392066955566, -1.4178828001022339, -2.4959769248962402, 1.9181007146835327, 2.5025038719177246, 3.2404286861419678, 2.980084180831909, 0.4806034564971924, -1.3657540082931519, -1.5836915969848633, -1.1155809164047241, -0.5741113424301147, -4.355008125305176, -2.7711706161499023, 0.9601165056228638, 3.070467948913574, 5.787967205047607, 1.4162689447402954, -2.0836620330810547, -0.23486563563346863, -0.9470235705375671, -0.1887872815132141, 2.8895058631896973, 0.8224835395812988, -1.180302381515503, -3.90937876701355, -1.9852757453918457, 2.713793992996216, 1.2686208486557007, 0.6188582181930542, 0.8236304521560669, -0.20369213819503784, -1.6914393901824951, 0.6971004009246826, 3.160860061645508, 3.406285047531128, -0.9962027072906494, -1.0968273878097534, 2.2172563076019287, 0.4662418067455292, -5.844573497772217, -3.8982057571411133, -1.311713457107544, 0.5464675426483154, 4.210468292236328, 2.9338767528533936, 2.1847527027130127, -2.2280831336975098, -4.5374555587768555, -0.3686383366584778, 2.9138858318328857, 2.85986590385437, -1.0497465133666992, -5.017723560333252, -1.3393824100494385, 5.159735679626465, 1.8502857685089111, -2.9600112438201904, -2.9922938346862793, 2.3142642974853516, 2.4811220169067383, -2.8490002155303955, -5.040712833404541, 2.2046360969543457, 4.648138046264648, 4.185016632080078, -0.4066442847251892, -6.094254970550537, -6.773036003112793, -1.5877573490142822, 2.4430646896362305, 3.676973819732666, 4.114546775817871, 3.359382152557373, -1.8726768493652344, -5.9611687660217285, -4.412135124206543, 1.8367202281951904, 4.762941360473633, 3.595273733139038, 0.21365046501159668, -0.6035733819007874, -0.9421162009239197, -0.5101281404495239, -1.7856829166412354, -1.5191340446472168, -1.3925564289093018, 0.009227752685546875, 3.821851968765259, 5.108097076416016, 0.859885036945343, -2.2075343132019043, -1.5512434244155884, -0.8496426343917847, -0.24691829085350037, 1.2934455871582031, 0.5056892037391663, -0.2199764847755432, -0.8858395218849182, -2.2358734607696533, 0.02085578441619873, 2.9166364669799805, 0.5469754338264465, 0.6554020047187805, 0.16938714683055878, -1.0381323099136353, -0.3424641788005829, -1.7935850620269775, -4.4846625328063965, -1.6463265419006348, 4.097928047180176, 7.3684797286987305, 1.4022471904754639, -2.7830193042755127, -5.533059120178223, -3.0407004356384277, -0.47185075283050537, 0.4405283033847809, 5.338811874389648, 7.5389604568481445, 0.09196019172668457, -5.875306606292725, -10.009699821472168, -6.529788970947266, -1.808809757232666, 2.4735541343688965, 6.418558597564697, 4.598551273345947, 3.8423924446105957, 2.8491737842559814, -1.407285213470459, -4.51643180847168, -5.076656341552734, -3.2968697547912598, 0.8537232875823975, 4.38569974899292, 0.9255545139312744, -0.7271426916122437, -0.6770359873771667, 0.20812149345874786, 0.1964731216430664, 2.3061938285827637, 2.615975856781006, -0.00115126371383667, -6.45416259765625, -5.914156436920166, -3.0136313438415527, 2.9213576316833496, 6.459775447845459, 3.5316755771636963, -1.3200438022613525, -1.1510069370269775, 4.60537576675415, 4.066685676574707, -2.257075786590576, -3.371087074279785, -3.0820441246032715, -1.2723939418792725, 0.21981698274612427, -2.2680275440216064, 0.06540346145629883, 2.719348430633545, 3.74233341217041, 4.524630546569824, -3.142453908920288, -4.083639621734619, -1.5606426000595093, -3.1001429557800293, -0.4348854124546051, 0.9439926147460938, 1.7482573986053467, 3.5671658515930176, 1.7588646411895752, 0.5046351552009583, -1.1346900463104248, -4.099084854125977, -4.351901531219482, -1.9243543148040771, 3.0471999645233154, 5.234930515289307, 3.251427173614502, -0.97147536277771, -5.830244064331055, -4.714704513549805, 3.1980679035186768, 6.955374717712402, 1.5290207862854004, -4.247560977935791, -4.583432674407959, 1.0236287117004395, 0.14148658514022827, -2.4174251556396484, -0.18482835590839386, -0.5459127426147461, 1.6404727697372437, 3.6492724418640137, 1.5891377925872803, -3.7392334938049316, -2.9750804901123047, -0.6596426963806152, 2.2652835845947266, -1.0906403064727783, 0.9411482214927673, -1.1602463722229004, 0.5944052934646606, 0.3919197618961334, 1.3354527950286865, -0.3737771511077881, -0.41875994205474854, -0.3328641653060913, 0.09088969230651855, -1.769518256187439, -2.6229376792907715, 0.2839857339859009, 5.451498031616211, 4.768214225769043, 0.02519664168357849, -2.025582790374756, -6.124444007873535, -3.913926839828491, -1.5979738235473633, 1.301677942276001, 5.35411262512207, 5.16655969619751, 0.110079824924469, -2.613542079925537, -3.406557083129883, -0.6456653475761414, 0.29516875743865967, 1.625256061553955, 2.8190505504608154, -0.8344273567199707, -1.4878456592559814, -0.1119229793548584, 0.2563098073005676, -3.052633762359619, -2.481753349304199, 1.1914905309677124, 2.7985286712646484, 1.7252745628356934, 0.2990577220916748, 0.4042443633079529, 1.159165859222412, -5.809576988220215, -5.4107890129089355, 1.4481544494628906, 1.3141024112701416, 2.082383394241333, 0.24207806587219238, -0.4707217216491699, 1.6598963737487793, 2.62138032913208, 1.1094820499420166, -2.1319589614868164, -2.9723663330078125, -0.3215544819831848, -0.5805666446685791, -0.5306975841522217, 1.3890855312347412, 0.19805508852005005, -3.6027815341949463, -3.579876184463501, 1.918858289718628, 3.7475733757019043, 1.9380073547363281, 2.5878734588623047, -0.2961309552192688, -2.006910562515259, -4.701722145080566, -3.074525833129883, 3.5867059230804443, 8.09969425201416, 3.3423781394958496, -3.1724133491516113, -6.2669196128845215, -0.223272442817688, 4.4244794845581055, 3.7984557151794434, -1.351547360420227, -1.6260930299758911, -1.1792067289352417, 1.9734361171722412, 2.8659024238586426, 2.2561678886413574, -2.3969643115997314, -4.222955226898193, 1.2772467136383057, 3.636972665786743, 1.5761890411376953, -0.019760042428970337, -5.500866889953613, -6.309943675994873, 1.3129260540008545, 7.153741836547852, 7.773553848266602, 1.4780527353286743, -3.1928582191467285, -3.679527521133423, -2.6147098541259766, -0.5624263882637024, 1.977541208267212, 0.8130916357040405, -2.122121810913086, -1.765398621559143, 3.698258399963379, 7.313661098480225, 5.020234107971191, -3.049678087234497, -8.335943222045898, -7.783458709716797, -1.7836689949035645, 4.84153413772583, 5.667159080505371, 1.7436672449111938, 0.7359061241149902, 2.021495819091797, -1.839647889137268, -2.004053831100464, 0.9426374435424805, 2.9943697452545166, -3.9547860622406006, -5.4470534324646, -0.3889191150665283, 2.900886058807373, 2.6694841384887695, -0.2472890317440033, -1.0872507095336914, 2.949375629425049, 1.2495899200439453, 1.4490420818328857, 0.594490647315979, -4.540374279022217, -3.772817373275757, 0.7593531608581543, 1.8387608528137207, -0.03918123245239258, -0.9630194902420044, 4.557796478271484, 5.504979610443115, -1.996469497680664, -4.512250900268555, -0.6805402040481567, 2.708136558532715, 1.6167092323303223, -0.8016555309295654, -2.7494847774505615, -3.897836923599243, -1.3994847536087036, 5.095283508300781, 4.98621129989624, 3.406505584716797, -3.1344096660614014, -4.66117000579834, -3.728477716445923, -1.4921739101409912, 4.858713626861572, 7.004875659942627, 0.1836000680923462, -1.6014792919158936, -5.483407497406006, -2.687830924987793, 2.697514295578003, 4.044711589813232, 2.098740816116333, -1.0371911525726318, -4.74155855178833, -0.17669260501861572, 0.2873210608959198, 4.679799556732178, 1.7263367176055908, -2.677999258041382, -2.471249580383301, 1.937028408050537, 1.7471506595611572, 0.8965490460395813, -1.2675113677978516, -1.279213547706604, -3.8283233642578125, -6.191581726074219, -2.99597430229187, 5.448976993560791, 10.57594108581543, 7.938887119293213, 0.2714526355266571, -9.999433517456055, -6.4778923988342285, 0.5412310361862183, 3.938380718231201, 2.047297239303589, -1.3852407932281494, -1.7070153951644897, -1.4676579236984253, -3.5352602005004883, -0.827677309513092, 0.1101405918598175, 4.182108402252197, 5.740665912628174, 1.6506081819534302, -0.2334877848625183, -1.9573372602462769, -3.842331647872925, 0.5357877016067505, 1.4106075763702393, 2.5094079971313477, 0.726189136505127, -1.5025511980056763, -1.2081936597824097, 0.14235559105873108, -1.2583696842193604, 1.3602244853973389, -0.047210097312927246, -1.0151416063308716, -2.693601369857788, 1.4575681686401367, 0.9461497068405151, 0.15662318468093872, 1.5345983505249023, 3.389875650405884, 0.2105887234210968, -2.5181989669799805, -3.7085888385772705, -0.6756264567375183, 1.9056577682495117, 2.9186251163482666, 0.2367788702249527, -0.8364704847335815, -2.5229830741882324, -1.9297956228256226, -0.7771542072296143, 0.9727861881256104, 3.0048723220825195, 4.922138214111328, 3.570409059524536, -1.1032028198242188, -7.524868965148926, -6.885793685913086, -0.639305591583252, 6.195074081420898, 5.714499473571777, -0.8414403796195984, -3.6756057739257812, -2.085979461669922, 4.001873016357422, 4.31843900680542, 4.318949222564697, 2.61130428314209, -3.195202112197876, -6.829565048217773, -2.7613584995269775, -2.3402533531188965, 0.8024928569793701, 1.9029737710952759, 5.463413715362549, 3.533153533935547, -3.76115083694458, -3.013094663619995, 2.286870241165161, 3.150444507598877, -1.1038869619369507, -5.03582239151001, 0.1822589635848999, 4.605978965759277, 1.4979438781738281, -0.7635763883590698, -1.667793869972229, -1.950809359550476, -2.284881591796875, -1.64835524559021, 1.7746944427490234, 2.0425353050231934, 2.049541473388672, 3.028715133666992, 2.214163303375244, -3.0436244010925293, -7.293829441070557, -2.4812557697296143, 1.9045870304107666, 4.8428955078125, 2.590439796447754, -0.5432276725769043, -1.9894088506698608, 0.4778033494949341, 1.1147816181182861, 0.7773457169532776, -4.668926239013672, -1.165113925933838, 0.9112159013748169, 3.6590754985809326, 2.8984434604644775, -0.8391789197921753, -0.38740074634552, -0.48997563123703003, -3.430260419845581, 0.4724094867706299, -2.1392526626586914, -1.331526517868042, 3.687488079071045, 3.873744487762451, 3.2205464839935303, -1.0815623998641968, -4.915101051330566, -3.0516462326049805, -0.3269244432449341, 1.107442855834961, 1.1999999284744263, 2.325935125350952, 2.934774875640869, 1.0302932262420654, -0.3195141553878784, -1.5271334648132324, -3.749661684036255, -5.7357940673828125, -4.473880290985107, 1.4136184453964233, 4.002243995666504, 4.3123674392700195, 4.472318172454834, -0.4422362446784973, -2.8629610538482666, -3.050614356994629, -2.4811573028564453], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(conv1d_same, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")